<a href="https://colab.research.google.com/github/speediedan/interpretune/blob/main/src/it_examples/notebooks/publish/circuit_tracer_examples/attention_attribution_analysis.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" />
</a>

In [ ]:
# Uncomment to run installation steps if you do not have a development
# editable install and want to run this notebook in a fresh environment.
# %pip install uv
# %uv pip install --upgrade pip setuptools wheel && \
# %uv pip install 'git+https://github.com/speediedan/interpretune.git@main[examples]'
# %uv pip install --group git-deps
#
# NOTE: This cell is intentionally commented out. We will uncomment these
# install commands once we no longer need to preserve editable installs
# for active developer venvs.


# Attention-Aware Attribution

An attribution graph holds the attention patterns and the norm denominators constant by default. That is what
makes its edges exact direct effects in a linearized model, and it is also a blind spot: an effect that acts by
changing *where attention looks* is credited to no node, and an edge does not say which head carried it.

This notebook demonstrates the three attention-aware additions to the analysis surface, each through its
top-level wrapper, on one prompt whose answer is easy to state:

- **The two regimes:** `compute_attribution_graph` with patterns frozen (the default) and with patterns live
  (`freeze_attention=False`). A live-pattern edge is a first-order derivative, not an exact direct effect.
- **Head loadings:** `it.head_loadings(...)` splits the edges into one target across the heads of a layer.
- **Head conductance:** `it.head_conductance(...)` credits each head for the measured change of a target
  between two prompts of the same length, split into a pattern (QK) part and a value (OV) part.

Alongside each report, the [analysis injection](../attribution_analysis/attribution_analysis.ipynb) framework
reads the intermediate values circuit-tracer builds it from: which locations each regime detaches, the
Neumann series behind node influence, the per-head copies behind a head loading, and the path a conductance
integral follows.

The nnsight model-level adapter is the first to support these reports; support in other model-level adapters
is planned.

In [ ]:
# @title Imports { display-mode: "form" }
import importlib.util
import json
import tempfile
from pathlib import Path
from pprint import pformat

import numpy as np
import torch

import interpretune as it
import interpretune.analysis  # registers the top-level op wrappers used below
from it_examples.seeds import ensure_local_seeds
from it_examples.utils.attention_attribution_helpers import (
    align_regime_nodes,
    check_next_token,
    display_plotly_figure,
    node_kinds,
    plot_conductance_path,
    plot_head_heatmap,
    plot_head_split,
    plot_neumann_convergence,
    plot_regime_edges,
    plot_residual_convergence,
    plot_ridgeline,
    plot_running_credit,
    print_next_token_checks,
    source_layers,
    top_k_overlap,
)
from it_examples.utils.example_helpers import required_os_env
from interpretune import ITSession, ITSessionConfig
from interpretune.config import AnalysisCfg, init_analysis_cfgs

## Notebook Parameters

This cell contains parameters that can be injected by papermill during parameterized test runs.
`conductance_steps` is the step count of the path integral in the conductance section: a path that moves
sharply needs many more steps than the default, which is exactly what its completeness residual reports.
`residual_sweep_steps` are the step counts the convergence check at the end compares.

In [ ]:
# Parameters - These will be injected by papermill during parameterized test runs
core_log_dir = None  # Directory to save analysis logs (if None, a temp directory will be created)
conductance_steps = 32  # Path-integral steps for head_conductance; read its residual before trusting few steps
head_layers = None  # Layers for the head-loadings report (None reports every layer)
residual_sweep_steps = [8, 16, 32]  # Step counts for the completeness-residual convergence check
enable_analysis_injection = True  # Read circuit-tracer's intermediate values through the analysis points
analysis_config_path = "attention_attribution_injection_config.yaml"  # Analysis injection config beside this notebook

In [ ]:
# @title Environment Setup { display-mode: "form" }
env_path: str | None = None  # set to '/full/path/to/.env' to override
os_env_reqs = None
assert required_os_env(env_path=env_path, env_reqs=os_env_reqs)

## Analysis Injection

The analysis points are patched into circuit-tracer before anything imports it, so they are set up first. Each
point records into a trace that the cells below read and reset; the points and what each reads are in
`attention_attribution_points.py`, and the lines they attach to are in `attention_attribution_injection_config.yaml`.
The [attribution analysis](../attribution_analysis/attribution_analysis.ipynb) notebook walks through the
framework itself.

In [ ]:
NOTEBOOK_DIR = Path.cwd()
config_path = Path(analysis_config_path)
config_path = (config_path if config_path.is_absolute() else NOTEBOOK_DIR / config_path).resolve()
log_dir = Path(core_log_dir).expanduser() if core_log_dir else Path(tempfile.gettempdir())
log_dir.mkdir(parents=True, exist_ok=True)

# the notebook reads the traces through the same module the orchestrator loads the points from
_points_spec = importlib.util.spec_from_file_location(
    "attention_attribution_points", config_path.parent / "attention_attribution_points.py"
)
points = importlib.util.module_from_spec(_points_spec)
_points_spec.loader.exec_module(points)

analysis_injector = None
if enable_analysis_injection:
    from interpretune.analysis.injection import orchestrator

    analysis_injector = orchestrator.setup_analysis_injection(
        config_path=config_path,
        target_package="circuit_tracer",
        config_overrides=f"settings:\n  log_dir: {log_dir.as_posix()}\n",
    )
    print(f"Analysis points active: {sorted(analysis_injector.config.file_hooks)}")
points.reset_trace()

## Session Configuration

We load the `speediedan/rte` component configuration for Gemma-2-2b via `it.hub.load`, with the nnsight
backend and the Gemma Scope transcoder set, the same substrate as the
[CT analysis backend demo](ct_analysis_backend_demo.ipynb).

In [ ]:
ensure_local_seeds()  # idempotent, offline: seed publish sources -> local components cache
base_itdm_cfg, base_it_cfg, dm_cls, m_cls = it.hub.load("speediedan/rte", "rte_demo.gemma2.circuit_tracer")

base_it_cfg.circuit_tracer_cfg.backend = "nnsight"
base_it_cfg.circuit_tracer_cfg.transcoder_set = "gemma"
base_it_cfg.circuit_tracer_cfg.verbose = False
if core_log_dir:
    base_it_cfg.core_log_dir = core_log_dir
print(pformat(base_it_cfg.circuit_tracer_cfg))

session_cfg = ITSessionConfig(
    adapter_ctx=(it.Adapter.core, it.Adapter.nnsight, it.Adapter.circuit_tracer),
    datamodule_cfg=base_itdm_cfg,
    module_cfg=base_it_cfg,
    datamodule_cls=dm_cls,
    module_cls=m_cls,
)
it_session = ITSession(session_cfg)
it.it_init(**it_session)

module = it_session.module
model = module.replacement_model
tokenizer = model.tokenizer
module.analysis_cfg = AnalysisCfg(target_op=it.compute_attribution_graph, ignore_manual=True, save_tokens=False)
init_analysis_cfgs(module, [module.analysis_cfg])
print(f"Module type: {type(module).__name__}")
print("Session initialized successfully!")

## One Prompt, One Swap, and a Sanity Check

The prompt is the capital prompt from circuit-tracer's tutorial and the backend demo: the answer, Austin,
needs one unstated step (Dallas is in Texas). It is kept so the graphs here stay comparable with upstream's.

The counterfactual swaps Dallas for **Detroit**, which `head_conductance` needs to be the same token length:
it integrates along the line between the two prompts' embeddings, so they must align position by position.
The swapped city is a large city that is *not* its state's capital, so the answer to the swapped prompt is a
different capital (Lansing) reached by the same unstated step. A capital city would confound the comparison:
for "the state containing Denver" the right answer is Denver itself, so copying the stated city and retrieving
a capital give the same token, and the measured change could not separate them.

The check below states what the comparison depends on before anything is attributed: on each prompt the
model's top answer is the expected capital, with a clear margin over the next token, and copying the stated
city is far less likely than answering.

In [ ]:
prompt = "Fact: the capital of the state containing Dallas is"
swapped_prompt = "Fact: the capital of the state containing Detroit is"
answer, swapped_answer = "▁Austin", "▁Lansing"

prompt_ids = tokenizer.encode(prompt)
swapped_ids = tokenizer.encode(swapped_prompt)
assert len(prompt_ids) == len(swapped_ids), "head_conductance needs two prompts of the same token length"
changed = [i for i, (a, b) in enumerate(zip(prompt_ids, swapped_ids, strict=True)) if a != b]
assert len(changed) == 1, f"the swap should change exactly one position, changed {changed}"
print(f"Prompt tokens: {[tokenizer.decode([t]) for t in prompt_ids]}")
print(
    f"Changed positions: {changed} ({[tokenizer.decode([prompt_ids[i]]) for i in changed]} -> "
    f"{[tokenizer.decode([swapped_ids[i]]) for i in changed]})"
)

checks = [
    check_next_token(model, tokenizer, "Dallas", prompt, answer, copy_token="▁Dallas"),
    check_next_token(model, tokenizer, "Detroit", swapped_prompt, swapped_answer, copy_token="▁Detroit"),
]
print_next_token_checks(checks)
for c in checks:
    assert c.rank == 1, f"{c.label}: the model must answer {c.expected!r} first, got {c.top[:3]}"
    assert c.margin >= 1.0, f"{c.label}: {c.expected!r} leads by only {c.margin:.2f} logits"
    assert c.copy_prob < 0.25 * c.prob, f"{c.label}: copying the stated city is too likely ({c.copy_prob:.3f})"

austin_id = tokenizer.encode(answer, add_special_tokens=False)[-1]
lansing_id = tokenizer.encode(swapped_answer, add_special_tokens=False)[-1]
print(f"Key tokens: Austin={austin_id}, Lansing={lansing_id}")

## The Two Regimes Disagreeing

The same graph computed twice: once with the default regime (patterns and norms frozen) and once with
patterns live. The regime is recorded on each graph, and the record matters: a graph whose record was lost
reads as frozen, which would mislabel a patterns-live graph with no error anywhere.

Under the live regime an edge is a first-order derivative of the target with respect to the source, read at
this prompt. It credits a source for changing where attention looks, at the price of exactness: the frozen
graph's edges are exact direct effects in the linearized model, and the live graph's are not.

The `ap_gradient_flow_regime` analysis point shows the mechanism: before the backward pass, circuit-tracer
detaches the attention-pattern outputs in the frozen regime and leaves them attached in the live one.

**Compare the two graphs node by node, not index by index.** Each graph selects its own feature nodes from the
prompt's active features, by attribution, so the two adjacency matrices have the same shape while their feature
columns name different features. `align_regime_nodes` maps both onto the features they both selected, plus the
error, token and logit nodes, which the two lay out identically; every comparison below reads those shared nodes.

In [ ]:
points.reset_trace(points.GRADIENT_FLOW)
frozen = it.compute_attribution_graph(
    module, None, None, 0, prompt=prompt, max_n_logits=5, desired_logit_prob=0.5, verbose=False
)
live = it.compute_attribution_graph(
    module,
    None,
    None,
    0,
    prompt=prompt,
    max_n_logits=5,
    desired_logit_prob=0.5,
    verbose=False,
    freeze_attention=False,
)

print(f"Frozen regime: {json.loads(frozen.graph_attribution_regime_json)}")
print(f"Live regime:   {json.loads(live.graph_attribution_regime_json)}")
target_probs = [round(float(p), 4) for p in frozen.logit_probabilities]
print(f"Logit targets: {frozen.logit_target_tokens} (probabilities {target_probs})")

if enable_analysis_injection:
    flows = points.read_trace(points.GRADIENT_FLOW)
    for flow in flows:
        print(
            f"  freeze_attention={flow['freeze_attention']!s:<5} detaches {flow['detached']} locations "
            f"({flow['attention_locations']} attention-pattern, {flow['norm_scale_locations']} norm-scale)"
        )
    regimes = {flow["freeze_attention"] for flow in flows}
    assert regimes == {True, False}, f"the gradient-flow point must see both regimes, saw {regimes}"

aligned = align_regime_nodes(frozen, live)
n_selected = len(frozen.selected_features)
same_index = int((frozen.selected_features == live.selected_features).sum())
print(
    f"Feature nodes: {n_selected} frozen, {len(live.selected_features)} live, {aligned.n_common} in both; "
    f"only {same_index} sit at the same index in both graphs"
)
frozen_adj = frozen.adjacency_matrix.index_select(0, aligned.first).index_select(1, aligned.first).float()
live_adj = live.adjacency_matrix.index_select(0, aligned.second).index_select(1, aligned.second).float()
delta = (live_adj - frozen_adj).abs()
print(f"Edges between shared nodes that change: {int((delta > 0).sum())} of {delta.numel()}")
assert float(delta.max()) > 0, "positive control: the two regimes must actually disagree"

Rows of the adjacency matrix are target nodes and columns are sources, in the order
features, error nodes, token (embed) nodes, logit nodes; the aligned matrices keep that order over the shared
features. The table reads the Austin logit row under both regimes, for the sources whose edge is largest in
either one.

In [ ]:
n_pos = len(frozen.input_tokens)
n_layers = model.cfg.n_layers
target_ids = [int(t) for t in frozen.logit_target_ids]
n_logits = len(target_ids)
n_common = aligned.n_common
n_aligned = len(aligned.first)
assert n_common + (n_layers + 1) * n_pos + n_logits == n_aligned, "unexpected node layout"


def other_node_label(k):
    # the k-th node after the features: error nodes layer-major, then token nodes, then logit nodes
    if k < n_layers * n_pos:
        return f"error L{k // n_pos} pos{k % n_pos}"
    k -= n_layers * n_pos
    if k < n_pos:
        return f"token pos{k} {tokenizer.decode([int(frozen.input_tokens[k])])!r}"
    return f"logit {frozen.logit_target_tokens[k - n_pos]!r}"


def aligned_label(j):
    if j < n_common:
        layer, pos, feat = (int(v) for v in aligned.common_features[j])
        return f"feature L{layer} pos{pos} #{feat}"
    return other_node_label(j - n_common)


assert austin_id in target_ids, f"Austin must be a logit target of the graph, got {frozen.logit_target_tokens}"
austin_row = n_aligned - n_logits + target_ids.index(austin_id)
frozen_row, live_row = frozen_adj[austin_row], live_adj[austin_row]

strength = torch.maximum(frozen_row.abs(), live_row.abs())
top_sources = strength.topk(8).indices.tolist()
print(f"{'source into Austin':<28} {'frozen':>10} {'live':>10}")
for src in top_sources:
    print(f"{aligned_label(src):<28} {float(frozen_row[src]):>10.4f} {float(live_row[src]):>10.4f}")

### Where the regimes disagree

Every source's edge into Austin, frozen against live. Points on the diagonal are edges the live patterns do not
change; the labelled points change most. The ridgeline below groups the same changes by the source's layer:
each curve is the distribution of $\lvert e_{\mathrm{live}} - e_{\mathrm{frozen}} \rvert$ over that layer's
sources, so a layer whose curve sits to the right has sources the frozen regime credits very differently.

In [ ]:
kinds = node_kinds(n_common, n_layers, n_pos, n_logits)
labels = [aligned_label(j) for j in range(n_aligned)]
display_plotly_figure(plot_regime_edges(frozen_row, live_row, kinds, labels, "Austin"), "it-regime-edges")

layer_of = source_layers(aligned.common_features, n_layers, n_pos, n_logits)
change = (live_row - frozen_row).abs().cpu().numpy()
series = [("tokens", change[layer_of == -1])] + [
    (f"L{layer}", change[layer_of == layer]) for layer in range(n_layers) if (layer_of == layer).any()
]
display_plotly_figure(
    plot_ridgeline(
        series, "How much each layer's edges into Austin change with live patterns", "log10 |live - frozen|"
    ),
    "it-regime-ridgeline",
)

### What the change does to node influence

Pruning ranks nodes by influence: the total weight of every path from a node to the logits, through the
row-normalized absolute adjacency $A$. circuit-tracer sums that path series term by term,

$$
w^{\mathsf{T}} (A + A^2 + A^3 + \cdots),
$$

with $w$ the logit probabilities, until a term is zero. The `ap_node_influence_*` points record every term, so the
two regimes' series can be compared directly, as the attribution-analysis notebook does for one graph. The overlap
of the most influential nodes says how much of the pruned graph the regime would change.

In [ ]:
# imported here, after the analysis points are patched in, so the name binds to the patched module
from circuit_tracer.graph import compute_node_influence


def logit_weights(batch):
    weights = torch.zeros(batch.adjacency_matrix.shape[0])
    weights[-n_logits:] = torch.as_tensor(batch.logit_probabilities, dtype=torch.float32)
    return weights


influence = {}
neumann = {}
for name, batch in (("frozen", frozen), ("live", live)):
    points.reset_trace(points.NEUMANN_RUNS)
    influence[name] = compute_node_influence(batch.adjacency_matrix.float().cpu(), logit_weights(batch))
    if enable_analysis_injection:
        runs = points.read_trace(points.NEUMANN_RUNS)
        assert len(runs) == 1 and len(runs[0]) > 1, "the Neumann points must record one multi-term series per call"
        neumann[name] = runs[0]
        # positive control: the recorded terms add up to the influence circuit-tracer returned
        assert torch.allclose(torch.stack(runs[0]).sum(0), influence[name], rtol=1e-4, atol=1e-6)
        print(f"{name}: {len(runs[0])} terms until the series vanishes")

# features are matched by identity (their row of active_features), since each graph indexes its own selection
n_frozen, n_live = len(frozen.selected_features), len(live.selected_features)
overlap = top_k_overlap(
    influence["frozen"][:n_frozen],
    influence["live"][:n_live],
    ks=(10, 25, 100),
    ids_a=frozen.selected_features.tolist(),
    ids_b=live.selected_features.tolist(),
)
print("Top-k feature overlap by influence, frozen vs live (Jaccard):", {k: round(v, 3) for k, v in overlap.items()})

if neumann:
    display_plotly_figure(plot_neumann_convergence(neumann), "it-neumann-regimes")
graph_kinds = {
    name: np.asarray(node_kinds(len(batch.selected_features), n_layers, n_pos, n_logits))
    for name, batch in (("frozen", frozen), ("live", live))
}
by_kind = []
for kind in ("feature", "error", "token"):
    for name in ("frozen", "live"):
        by_kind.append((f"{kind}s, {name}", influence[name].numpy()[graph_kinds[name] == kind]))
display_plotly_figure(
    plot_ridgeline(by_kind, "Node influence by node kind, under each regime", "log10 influence"), "it-influence-kinds"
)

## Head Loadings

`head_loadings` splits the edges into one target across the heads of each requested layer, plus a bypass
share for the part of those edges no head carries. Each number is a share: the head's part of the summed
edges into the target at that layer. That sum is the share's denominator, and it is the thing to look at
before quoting a share: where the summed edges at a layer are small, the same absolute attribution becomes
a large share, and small measurement differences move it a lot. The partition check beside the table is the
internal control: the head shares and the bypass must sum back to the whole, up to the reported partition
error.

The split comes from one backward pass over several copies of the prompt: in copy $h$ a stop-gradient leaves only
head $h$ of the layer live, one copy keeps no head (the bypass), and one keeps every head (the edge). The
`ap_head_loadings_rows` point records those copies, so one source's edge can be rebuilt from its parts below.

In [ ]:
points.reset_trace(points.HEAD_LOADING_ROWS)
loadings = it.head_loadings(module, None, None, 0, prompt=prompt, head_target=answer, head_layers=head_layers)
layers_reported = loadings.head_loading_layers.tolist()

partitioned = loadings.head_loading_shares.sum(1) + loadings.head_loading_bypass_shares
print(f"Summed edges into the target (the shares' denominator): {float(loadings.head_loading_total):.4f}")
print(f"{'layer':>5} {'top head':>8} {'share':>8} {'bypass':>8} {'partition err':>13}")
for row, layer in enumerate(layers_reported):
    shares = loadings.head_loading_shares[row]
    top_head = int(shares.abs().argmax())
    print(
        f"{layer:>5} {top_head:>8} {float(shares[top_head]):>8.3f} "
        f"{float(loadings.head_loading_bypass_shares[row]):>8.3f} "
        f"{float(loadings.head_loading_partition_errors[row]):>13.2e}"
    )

print(f"Partition check (shares + bypass, should read 1 per layer): {[round(float(v), 4) for v in partitioned]}")
assert bool(((partitioned - 1).abs() <= loadings.head_loading_partition_errors + 0.01).all()), (
    "head parts must partition the edges up to the reported partition error"
)
assert float(loadings.head_loading_shares.abs().max()) > 1e-3, "a report of zeros would also partition the edges"

display_plotly_figure(
    plot_head_heatmap(
        loadings.head_loading_shares,
        layers_reported,
        "Each head's share of the summed edges into Austin",
        "share",
        extra_column=("bypass - 1", loadings.head_loading_bypass_shares - 1),
    ),
    "it-head-loadings",
)

In [ ]:
if enable_analysis_injection:
    # head-loading rows run over the attribution context's sources: every ACTIVE feature, then errors, then tokens
    n_active = len(frozen.active_features)

    def context_label(idx):
        if idx < n_active:
            layer, pos, feat = (int(v) for v in frozen.active_features[idx])
            return f"feature L{layer} pos{pos} #{feat}"
        return other_node_label(idx - n_active)

    records = {record["layer"]: record for record in points.read_trace(points.HEAD_LOADING_ROWS)}
    assert sorted(records) == sorted(layers_reported), "one recorded set of copies per reported layer"
    assert all(r["rows"].shape[1] == n_active + (n_layers + 1) * n_pos for r in records.values()), "row layout"
    total = float(loadings.head_loading_total)
    for row, layer in enumerate(layers_reported):
        rows, n_heads = records[layer]["rows"], records[layer]["n_heads"]
        # positive control: the recorded copies reproduce the report's shares
        rebuilt = (rows[:n_heads] - rows[n_heads]).sum(1) / total
        assert torch.allclose(rebuilt, loadings.head_loading_shares[row].float(), atol=1e-3), f"layer {layer}"
    layer = layers_reported[int(loadings.head_loading_shares.abs().amax(1).argmax())]
    record = records[layer]
    source = int(record["rows"][-1].abs().argmax())
    print(f"Layer {layer} carries the largest head share; its largest source is {context_label(source)}")
    display_plotly_figure(plot_head_split(record, source, context_label(source), "Austin"), "it-head-split")

## Head Conductance

Conductance answers a different question: not what the graph credits on one prompt, but which heads carry a
*measured change* between two prompts. It integrates each head's pattern (QK) and value (OV) contribution
along the straight path $e(\alpha) = e_0 + \alpha (e_1 - e_0)$ between the Dallas and Detroit embeddings, for the
Austin-versus-Lansing logit difference. Passing the swapped position as `key_positions` also reports the part of
each head's credit that goes through that one key: attention moving toward or away from the city, or the city's
values changing.

Read `head_conductance_completeness_residual` before any number below it. The residual is what the integral
failed to account for at each layer. While it comes from the integral's discretization it falls quadratically
with the step count, so a large residual at the default step count is an instruction to raise
`conductance_steps`, not a result; the last section checks where it stops falling. `head_conductance_first_order` is the one-point linearization at the first prompt: where it disagrees
with the measured change, it is the case for integrating along the path at all. The `ap_conductance_*` points
record the target at every path point and each head's running credit, which the figures below draw.

In [ ]:
target_weights = torch.zeros(int(frozen.graph_vocab_size))
target_weights[austin_id], target_weights[lansing_id] = 1.0, -1.0

points.reset_trace(points.CONDUCTANCE_POINTS, points.CONDUCTANCE_RUNNING)
conductance = it.head_conductance(
    module,
    None,
    None,
    0,
    prompt=prompt,
    counterfactual_prompt=swapped_prompt,
    target_weights=target_weights,
    n_steps=conductance_steps,
    key_positions=changed,
)

residual = conductance.head_conductance_completeness_residual.abs()
print(
    f"Completeness residual: largest {float(residual.max()):.4f} at layer {int(residual.argmax())} "
    f"({conductance_steps} steps)"
)
t_from, t_to = float(conductance.head_conductance_t_from), float(conductance.head_conductance_t_to)
print(f"Target (Austin - Lansing logit): Dallas prompt {t_from:.4f}, Detroit prompt {t_to:.4f}")
print(
    f"Measured change: {t_to - t_from:.4f}; "
    f"first-order prediction at the Dallas prompt: {float(conductance.head_conductance_first_order):.4f}"
)
assert t_from > 0 > t_to, "each prompt's own capital must lead the target on that prompt"

for name, table, keyed in (
    ("pattern (QK)", conductance.head_conductance_qk, conductance.head_conductance_qk_keys),
    ("value (OV)", conductance.head_conductance_ov, conductance.head_conductance_ov_keys),
):
    flat = table.abs().flatten().topk(5)
    heads = []
    for i in flat.indices.tolist():
        layer, head = divmod(i, table.shape[1])
        heads.append((layer, head, round(float(table[layer, head]), 3), round(float(keyed[layer, head]), 3)))
    print(f"Top {name} heads (layer, head, credit, of which through the swapped city): {heads}")
assert float(conductance.head_conductance_qk.abs().max()) > 0 and float(conductance.head_conductance_ov.abs().max()) > 0

all_layers = list(range(n_layers))
display_plotly_figure(
    plot_head_heatmap(conductance.head_conductance_qk, all_layers, "Pattern (QK) conductance per head", "credit"),
    "it-conductance-qk",
)
display_plotly_figure(
    plot_head_heatmap(conductance.head_conductance_ov, all_layers, "Value (OV) conductance per head", "credit"),
    "it-conductance-ov",
)

In [ ]:
if enable_analysis_injection:
    path_points = points.read_trace(points.CONDUCTANCE_POINTS)
    # positive control: the recorded path starts and ends at the report's two targets
    assert abs(float(path_points[0]["t"][0]) - t_from) < 1e-3 and abs(float(path_points[-1]["t"][-1]) - t_to) < 1e-3
    assert abs(float(path_points[0]["dt_dalpha"][0]) - float(conductance.head_conductance_first_order)) < 1e-3
    display_plotly_figure(plot_conductance_path(path_points, "Austin - Lansing"), "it-conductance-path")
    display_plotly_figure(
        plot_running_credit(points.read_trace(points.CONDUCTANCE_RUNNING), conductance_steps), "it-conductance-running"
    )

### How many steps the path needs

The completeness residual falls as $1/n^2$ in the step count $n$ while the integral's discretization is what
it measures. Rerunning the integral at a few step counts shows which regime the default sits in. If the residual
stops falling, the remainder is not the path any more but the arithmetic: this model runs in bf16, whose
spacing near a logit of 8 is $8 \cdot 2^{-7} \approx 0.06$, so a floor of about a percent of a 15-logit change is
expected, and more steps will not lower it.


In [ ]:
sweep = sorted(set(residual_sweep_steps))
largest = []
for steps in sweep:
    points.reset_trace(points.CONDUCTANCE_POINTS, points.CONDUCTANCE_RUNNING)
    report = it.head_conductance(
        module,
        None,
        None,
        0,
        prompt=prompt,
        counterfactual_prompt=swapped_prompt,
        target_weights=target_weights,
        n_steps=steps,
    )
    largest.append(float(report.head_conductance_completeness_residual.abs().max()))
    print(f"{steps:>4} steps: largest completeness residual {largest[-1]:.5f}")
assert largest[-1] < largest[0], "the residual must fall as the step count rises"
print(
    f"From {sweep[-2]} to {sweep[-1]} steps the residual falls by {largest[-2] / largest[-1]:.2f}x "
    f"(a pure discretization error would fall {(sweep[-1] / sweep[-2]) ** 2:.0f}x)"
)
display_plotly_figure(plot_residual_convergence(sweep, largest), "it-residual-convergence")

## Summary

| Surface | What it showed here |
|----|---------|
| `compute_attribution_graph` regimes | Each regime selects its own feature nodes, so the graphs are compared on the nodes both select; there the Austin edges differ between frozen and live patterns, and each graph records which regime built it. The analysis points show the regimes differ by the detached attention-pattern outputs, and the change reaches node influence and so pruning. A live edge is a first-order derivative, not an exact direct effect. |
| `head_loadings` | The edges into Austin split across heads per layer, with the partition check read beside the shares and the summed edges named as the shares' denominator. The recorded per-head copies rebuild the shares and one source's edge. |
| `head_conductance` | The Dallas-to-Detroit change of the Austin-versus-Lansing logit difference, credited to heads in QK and OV parts, with the completeness residual read first and its convergence in the step count checked. |

The nnsight model-level adapter is the first to support this surface; support in other model-level adapters is
planned. The frozen-regime graph machinery behind the first section is the subject of the
[CT analysis backend demo](ct_analysis_backend_demo.ipynb), and gradient- and ablation-based attribution live in the
[attribution analysis](../attribution_analysis/attribution_analysis.ipynb) notebook.